# Analise Exploratoria de Dados (EDA) — Olist Intelligence Platform

Esta e a etapa formal de **Data Understanding** (CRISP-DM) / **Explore** (SEMMA) /
**Selection + Preprocessing overview** (KDD) do projeto — antes tinha sido feita de
forma implicita atraves das queries SQL em `analytics/sql/`, mas nao existia como uma
etapa isolada e documentada. Este notebook fecha essa lacuna.

**Fonte dos dados**: por padrao le a amostra estratificada em
`analytics/sample_fct_delivery_performance.csv` (10.000 pedidos, mesma proporcao de
atraso do dataset completo: ~8,1%) para que qualquer pessoa consiga rodar este notebook
sem precisar subir o Postgres. Para rodar contra o banco completo, troque a flag
`USE_LIVE_DB` na celula de configuracao abaixo.

**O que este notebook NAO faz**: engenharia de features definitiva (isso fica no dbt,
em `transformation/models/`) ou selecao final de modelo (isso fica em `ml/train.py`).
O objetivo aqui e puramente entender a forma dos dados antes de qualquer decisao de
modelagem — exatamente o proposito da fase de "Data Understanding".

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

USE_LIVE_DB = False  # True para consultar o Postgres ao vivo em vez da amostra local

if USE_LIVE_DB:
    from sqlalchemy import create_engine

    engine = create_engine(
        f"postgresql+psycopg2://{os.environ.get('POSTGRES_USER', 'olist_user')}:"
        f"{os.environ.get('POSTGRES_PASSWORD', 'olist_pass')}@"
        f"{os.environ.get('POSTGRES_HOST', 'localhost')}:"
        f"{os.environ.get('POSTGRES_PORT', '5432')}/"
        f"{os.environ.get('POSTGRES_DB', 'olist')}"
    )
    df = pd.read_sql("select * from marts.fct_delivery_performance", engine)
else:
    df = pd.read_csv(
        "sample_fct_delivery_performance.csv",
        parse_dates=["order_purchase_timestamp"],
    )

print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
df.head()

## 1. Visao geral: shape, tipos e valores ausentes

Primeiro passo de qualquer EDA: confirmar que os dados tem a forma esperada e mapear
onde estao os valores nulos antes de qualquer analise mais profunda.

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) == 0:
    print("Nenhum valor ausente nas colunas selecionadas.")
else:
    print((missing / len(df) * 100).round(2).astype(str) + "%")

**Observacao**: `review_score` e a coluna com mais ausencia esperada — nem todo
pedido entregue recebe uma avaliacao do cliente. Isso e tratado no dbt via inner join
(`int_order_reviews_joined.sql`), entao o mart final ja exclui pedidos sem review.

## 2. Estatisticas descritivas das variaveis numericas

In [ ]:
numeric_cols = ["delay_days", "total_delivery_days", "estimated_delivery_days",
                "order_total_value", "avg_freight_value", "n_items",
                "payment_installments", "review_score"]
df[numeric_cols].describe().T

## 3. Distribuicao da variavel-alvo (`is_delayed`)

Essa e a checagem mais importante antes de qualquer modelagem: **o dataset e
desbalanceado?** Essa resposta muda toda a estrategia de treino (ver
`ml/model_card.md`, secao sobre a armadilha do desbalanceamento).

In [ ]:
delay_rate = df["is_delayed"].mean()
print(f"Taxa de atraso: {delay_rate:.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
df["is_delayed"].value_counts().sort_index().plot(
    kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"]
)
ax.set_xticklabels(["No prazo", "Atrasado"], rotation=0)
ax.set_ylabel("Numero de pedidos")
ax.set_title(f"Distribuicao da variavel-alvo (taxa de atraso: {delay_rate:.1%})")
plt.tight_layout()
plt.show()

**Conclusao**: com ~8% de positivos, este e um problema de classificacao
desbalanceado. Isso justifica diretamente a decisao em `ml/train.py` de usar
`sample_weight` balanceado e escolher o melhor modelo por F1 (nao por AUC) — uma
metrica de acuracia simples aqui enganaria (prever sempre "nao atrasa" ja acerta 92%).

## 4. Distribuicao do atraso em dias (`delay_days`)

Quantos dias de atraso (ou adiantamento) os pedidos costumam ter?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
clipped = df["delay_days"].clip(-15, 30)  # corta outliers extremos so para visualizacao
sns.histplot(clipped, bins=45, ax=ax, color="#3498db")
ax.axvline(0, color="black", linestyle="--", linewidth=1, label="Prazo estimado")
ax.set_xlabel("Dias de atraso (negativo = adiantado)")
ax.set_title("Distribuicao do atraso em dias (valores extremos truncados em [-15, 30])")
ax.legend()
plt.tight_layout()
plt.show()

print(df["delay_days"].describe())

## 5. Nota de review vs. situacao da entrega

Esse e o achado de negocio central do projeto — vale confirmar aqui, na EDA, antes de
qualquer modelagem, que a hipotese realmente aparece nos dados brutos.

In [ ]:
order = ["entregue_adiantado", "no_prazo", "atraso_leve", "atraso_grave"]
summary = (
    df.groupby("delay_bucket")["review_score"]
    .agg(["mean", "count"])
    .reindex(order)
    .rename(columns={"mean": "nota_media", "count": "n_pedidos"})
)
print(summary.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="delay_bucket", y="review_score", order=order, ax=ax, palette="RdYlGn_r")
ax.set_xlabel("Situacao da entrega")
ax.set_ylabel("Nota do review")
ax.set_title("Nota de review por situacao da entrega")
plt.tight_layout()
plt.show()

**Conclusao**: a nota media cai de forma clara e monotonica conforme o atraso
aumenta — confirma a hipotese de negocio central do projeto (ver README, secao "O
problema de negocio") diretamente nos dados brutos, antes de qualquer transformacao.

## 6. Taxa de atraso por estado do cliente

Existe um padrao geografico? Isso tambem informa diretamente o `ml/monitoring/fairness_check.py`,
que verifica se o modelo tem desempenho consistente entre estados.

In [ ]:
by_state = (
    df.groupby("customer_state")
    .agg(taxa_atraso=("is_delayed", "mean"), n_pedidos=("is_delayed", "count"))
    .query("n_pedidos >= 10")
    .sort_values("taxa_atraso", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
by_state["taxa_atraso"].plot(kind="bar", ax=ax, color="#e67e22")
ax.set_ylabel("Taxa de atraso")
ax.set_xlabel("Estado do cliente")
ax.set_title("Taxa de atraso por estado (estados com >= 20 pedidos na amostra)")
ax.axhline(delay_rate, color="black", linestyle="--", linewidth=1, label="Media geral")
ax.legend()
plt.tight_layout()
plt.show()

by_state.head(10)

## 7. Correlacao entre variaveis numericas

Existe multicolinearidade entre as features candidatas? Relevante para interpretar
coeficientes do modelo baseline (regressao logistica).

In [ ]:
corr_cols = ["delay_days", "total_delivery_days", "estimated_delivery_days",
             "order_total_value", "avg_freight_value", "n_items",
             "payment_installments", "is_delayed"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlacao entre variaveis numericas")
plt.tight_layout()
plt.show()

**Observacao**: `total_delivery_days` e `delay_days` sao naturalmente
correlacionadas (ambas derivam do mesmo evento de entrega) — por isso `ml/train.py`
usa `estimated_delivery_days` como feature (conhecida no momento da compra), e NUNCA
`total_delivery_days` ou `delay_days`, que so existem depois que o pedido ja foi
entregue (usar essas features seria vazamento de dados/data leakage).

## 8. Distribuicao de categorias de produto e forma de pagamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["primary_product_category"].value_counts().head(10).plot(
    kind="barh", ax=axes[0], color="#9b59b6"
)
axes[0].set_title("Top 10 categorias de produto")
axes[0].invert_yaxis()

df["payment_type"].value_counts().plot(kind="bar", ax=axes[1], color="#1abc9c")
axes[1].set_title("Forma de pagamento")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 9. Outliers em valor do pedido e frete

Existem pedidos com valores extremos que podem distorcer o treino?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(y=df["order_total_value"].clip(upper=1000), ax=axes[0], color="#f39c12")
axes[0].set_title("Valor do pedido (truncado em R$1000 p/ visualizacao)")

sns.boxplot(y=df["avg_freight_value"].clip(upper=150), ax=axes[1], color="#e74c3c")
axes[1].set_title("Valor do frete (truncado em R$150 p/ visualizacao)")

plt.tight_layout()
plt.show()

print("Percentil 99 do valor do pedido:", df["order_total_value"].quantile(0.99).round(2))
print("Percentil 99 do frete:", df["avg_freight_value"].quantile(0.99).round(2))

**Decisao**: os outliers identificados sao valores altos porem plausiveis
(pedidos grandes/caros de verdade, nao erros de digitacao ou dados corrompidos) — por
isso `ml/train.py` nao aplica nenhum corte/clip nessas variaveis antes de treinar.
Modelos baseados em arvore (Gradient Boosting) sao naturalmente robustos a esses
outliers; a regressao logistica usa `StandardScaler`, que e mais sensivel a eles, mas
como o baseline serve principalmente de referencia de comparacao, isso foi aceito
conscientemente (ver `ml/model_card.md`).

## 10. Resumo dos achados que alimentam as proximas etapas

| Achado | Onde isso e usado depois |
|---|---|
| Dataset desbalanceado (~8% de atraso) | `ml/train.py` usa `sample_weight` balanceado e seleciona o modelo por F1 |
| Atraso derruba a nota de review de forma clara | Confirma a pergunta de negocio central (README) |
| Existe padrao geografico na taxa de atraso | `customer_state` vira feature categorica; `ml/monitoring/fairness_check.py` monitora isso apos o treino |
| `delay_days`/`total_delivery_days` vazam informacao do futuro | Somente `estimated_delivery_days` (conhecida na compra) e usada como feature |
| Outliers de valor sao legitimos, nao erros | Nenhum corte aplicado; efeito mitigado pela escolha de modelo baseado em arvore |

Proximo passo no pipeline: `transformation/models/` (dbt) formaliza essas transformacoes,
e `ml/train.py` faz a modelagem propriamente dita.